# Predictive Maintenance — Exploratory Data Analysis

This notebook explores the synthetic sensor dataset to confirm:
1. Feature distributions look realistic
2. Class imbalance matches industrial reality (~3-5% failure rate)
3. Features have meaningful relationships with the failure label

Findings from this EDA directly informed the modeling choices in `src/models/classifier.py`
(batch norm + dropout) and the training loop (pos_weight in BCE loss to handle imbalance).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.generate_data import generate_sensor_data, FEATURE_COLUMNS, TARGET_COLUMN

sns.set_style('whitegrid')
df = generate_sensor_data(n_samples=50_000, seed=42)
print(f'Shape: {df.shape}')
print(f'Failure rate: {df[TARGET_COLUMN].mean():.2%}')
df.head()

## 1. Class balance
Severe imbalance — about 1 in 25 samples is a failure. This motivates `pos_weight` in BCE loss.

In [ ]:
counts = df[TARGET_COLUMN].value_counts()
print(counts)
counts.plot(kind='bar', color=['steelblue', 'crimson'])
plt.title('Class distribution')
plt.xticks([0, 1], ['Healthy', 'Failure'], rotation=0)
plt.ylabel('Count')

## 2. Feature distributions by class
If there's no separation here, the model has nothing to learn. For each sensor,
we'd expect the failure class to have a shifted (usually higher) distribution.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.flat, FEATURE_COLUMNS):
    for label, color in [(0, 'steelblue'), (1, 'crimson')]:
        sns.kdeplot(df[df[TARGET_COLUMN] == label][col], ax=ax, label=f'class {label}', color=color)
    ax.set_title(col)
    ax.legend()
for ax in axes.flat[len(FEATURE_COLUMNS):]:
    ax.axis('off')
plt.tight_layout()

## 3. Feature correlation
Correlated features hurt linear models but MLPs handle them fine. Still worth
looking at to understand which sensors carry redundant information.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[FEATURE_COLUMNS + [TARGET_COLUMN]].corr(),
            annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Feature correlation matrix')

## 4. Baseline: logistic regression
Sanity check — a simple model should already beat random (AUC > 0.5).
The PyTorch MLP should beat this comfortably; if it doesn't, something is wrong.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

X = df[FEATURE_COLUMNS].values
y = df[TARGET_COLUMN].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler().fit(X_tr)
clf = LogisticRegression(class_weight='balanced', max_iter=1000)
clf.fit(scaler.transform(X_tr), y_tr)
probs = clf.predict_proba(scaler.transform(X_te))[:, 1]
print(f'Logistic regression baseline AUC: {roc_auc_score(y_te, probs):.4f}')